In [ ]:

# =================================================================================================
# NIH PATCH ABLATION — PIPELINE-4-LIKE MULTILABEL AUROC SETTING
# SINGLE ARM: P3_NoGradCAMScore
# =================================================================================================
#
# This notebook intentionally matches the pathology-head/evaluation SETTING of the user's
# uploaded Pipeline 4 as closely as possible while keeping the experiment a PATCH-SELECTION ABLATION.
#
# Matched Pipeline-4-style settings:
#   - five independent pathology outputs (NOT 5-way softmax)
#   - genuine multilabel labels; multi-positive cases retained
#   - unknown labels masked class-wise
#   - 6 patches per image
#   - image score = logsumexp(patch logits) - log(K)
#   - masked BCE-with-logits + 0.40 pairwise AUROC-ranking loss
#   - clean torchvision ImageNet-1K ViT-B/16 representation
#   - final 2 transformer blocks + final encoder layer norm trainable
#   - backbone LR 1e-5; pathology-head LR 3e-4; AdamW weight decay 1e-2
#   - 4 training epochs
#   - ImageNet resize/normalization and the same small RandomAffine training augmentation
#   - macro AUROC = mean of per-pathology binary AUROCs
#
# Deliberately NOT copied from full Pipeline 4:
#   - no quantum branch, prototypes, few-shot support, or fusion
#   - CheXzero is used only to define/select patches because this is a patch-selection ablation
#
# Primary endpoint:
#   IMAGE-LEVEL five-pathology multilabel macro AUROC/AUPRC after 6-patch MIL aggregation.
#
# Secondary endpoint:
#   weak patch-level AUROC/AUPRC using inherited image labels. This is NOT localization accuracy.
#
# All six split notebooks use the SAME cohort rules, SAME model, SAME optimizer, SAME seeds,
# SAME validation/test patch miner, and SAME metrics. Only the requested ablation arm differs.
# =================================================================================================

import os, sys, re, gc, math, json, time, random, hashlib, subprocess, platform, pickle
from pathlib import Path
from collections import defaultdict
from contextlib import nullcontext

RUN_START_TIME = time.perf_counter()
ARM_NAME = 'P3_NoGradCAMScore'
ARM_CONFIG = {'use_report': True, 'use_semantic': True, 'use_probability': True, 'use_gradcam': False, 'random_matched': False, 'whole': False}
WHOLE_IMAGE_ARM = bool(ARM_CONFIG["whole"])
NEEDS_CHEXZERO = not WHOLE_IMAGE_ARM

# Exact public OpenAI CLIP source revision used for CheXzero fallback installation.
CLIP_GIT_COMMIT = "d05afc436d78f1c48dc0dbf8e5980a9d471f35f6"

def _pip_install(spec):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])

try:
    import clip
except Exception:
    if NEEDS_CHEXZERO:
        _pip_install(f"git+https://github.com/openai/CLIP.git@{CLIP_GIT_COMMIT}")
        import clip
    else:
        clip = None

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.models import vit_b_16, ViT_B_16_Weights

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
)

# -------------------------------------------------------------------------------------------------
# CONFIG — Pipeline 4 pathology-head settings
# -------------------------------------------------------------------------------------------------
SEED_EXTRACT = 314159
FINAL_SEEDS = [42, 123, 777, 2027, 31415]

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU REQUIRED. Enable a Kaggle GPU accelerator.")

DEVICE = torch.device("cuda:0")
USE_CUDA = True
torch.cuda.set_device(DEVICE)
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

TARGET_CLASSES = [
    "Atelectasis",
    "Cardiomegaly",
    "Consolidation",
    "Edema",
    "Pleural Effusion",
]
N_CLASSES = len(TARGET_CLASSES)

Y_COLS = [f"_y{i}" for i in range(N_CLASSES)]
M_COLS = [f"_m{i}" for i in range(N_CLASSES)]
PATCH_Y_COLS = [f"target_{i}" for i in range(N_CLASSES)]
PATCH_M_COLS = [f"known_{i}" for i in range(N_CLASSES)]

CSV_DIR = "/kaggle/input/combinedreportimgpath"
TRAIN_CSV = os.path.join(CSV_DIR, "train_pairs_labeled.txt")
VAL_CSV = os.path.join(CSV_DIR, "val_pairs_labeled.txt")
TEST_CSV = os.path.join(CSV_DIR, "test_pairs_labeled.txt")

NIH_IMAGE_DIR = "/kaggle/input/datasets/nih-chest-xrays/data"

REPORT_DIR = "/kaggle/input/datasets/anikazarin/nih-reports/reports"
TRAIN_REPORT_DIR = os.path.join(REPORT_DIR, "train")
VAL_REPORT_DIR = os.path.join(REPORT_DIR, "val")
TEST_REPORT_DIR = os.path.join(REPORT_DIR, "test")

CHEXZERO_CKPT = (
    "/kaggle/input/models/anikazarin/chexzero/pytorch/default/1/"
    "best_64_5e-05_original_22000_0.864.pt"
)

OUT_DIR = Path(f"/kaggle/working/nih_patch_pipeline4like_v3_coordfix_{ARM_NAME}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep cohort identical across every arm.
MAX_TRAIN_IMAGES = 5000
MAX_VAL_IMAGES = 1000
MAX_TEST_IMAGES = 1000

# Patch selector
WORK_IMAGE_SIZE = 512
PATCH_SCALES = [64, 128, 256]
EVAL_PATCH_SCALES = [64, 128, 256]
STRIDE_FRAC = 0.50
MAX_CANDIDATES_PER_IMAGE = 160
PATCH_ENCODE_BATCH = 128
TOPK_PATCHES_PER_IMAGE = 6
NMS_IOU_THRESHOLD = 0.50

W_SEMANTIC = 0.45
W_PROBABILITY = 0.35
W_GRADCAM = 0.20
BINARY_PROB_TEMP = 14.0

# Exact target-head optimization values visible in Pipeline 4.
UNFREEZE_BLOCKS = 2
SOURCE_EPOCHS = 4
BACKBONE_LR = 1e-5
HEAD_LR = 3e-4
BATCH_PATCH_BUDGET = 32
IMAGE_BATCH_SIZE = max(1, BATCH_PATCH_BUDGET // TOPK_PATCHES_PER_IMAGE)  # 5 images x 6 patches = 30
EVAL_IMAGE_BATCH_SIZE = IMAGE_BATCH_SIZE
NUM_WORKERS = 2
WEIGHT_DECAY = 1e-2
TARGET_AUC_RANK_WEIGHT = 0.40
MAX_RANK_PAIRS_PER_CLASS = 4096
AMP = True

BOOTSTRAP_REPS = 1200
BOOTSTRAP_SEED = 20260824

CKPT_MAX_MISSING = 0
CKPT_MAX_UNEXPECTED = 0

TASK_DEFINITION = (
    "NIH five-pathology multilabel patch classification; multi-positive cases retained"
)
AUROC_DEFINITION = (
    "standard per-pathology binary AUROC on independent pathology scores; macro mean over five classes"
)
IMAGE_AGGREGATION = "logsumexp_patch_logits_minus_logK"
PATCH_METRIC_SCOPE = (
    "Patch-level AUROC/AUPRC inherit image labels and are weak-label diagnostics only; "
    "they are not lesion-localization metrics."
)
EXPERIMENT_SCHEMA = "nih_patch_pipeline4like_multilabel_v3_coordfix"

def validate_required_inputs():
    required = [TRAIN_CSV, VAL_CSV, TEST_CSV]
    if NEEDS_CHEXZERO:
        required.append(CHEXZERO_CKPT)

    missing = [p for p in required if not os.path.isfile(p)]

    if not os.path.isdir(NIH_IMAGE_DIR):
        raise FileNotFoundError(
            f"NIH image directory not found: {NIH_IMAGE_DIR}"
        )

    if not os.path.isdir(REPORT_DIR):
        raise FileNotFoundError(
            f"NIH report directory not found: {REPORT_DIR}"
        )

    if missing:
        raise FileNotFoundError(
            "Required Kaggle input(s) missing:\n  - "
            + "\n  - ".join(missing)
        )

    print("Kaggle input preflight: OK")
    print("ARM:", ARM_NAME)
    print("NIH_IMAGE_DIR:", NIH_IMAGE_DIR)
    print("REPORT_DIR:", REPORT_DIR)

validate_required_inputs()

def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED_EXTRACT)

print("=" * 110)
print("DEVICE:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print("ARM:", ARM_NAME, ARM_CONFIG)
# Regression test for the historical y1/y2 coordinate collision.
_reserved_crop_fields = {"x1", "y1", "x2", "y2", "scale", "rank"}
_patch_label_fields = set(PATCH_Y_COLS) | set(PATCH_M_COLS)
_namespace_collision = sorted(_reserved_crop_fields & _patch_label_fields)
if _namespace_collision:
    raise RuntimeError(
        f"Patch label names collide with crop-coordinate fields: {_namespace_collision}"
    )
print("Patch label/coordinate namespace self-test: OK")

print("Pipeline-4-like pathology setting:")
print(
    f"K={TOPK_PATCHES_PER_IMAGE}, epochs={SOURCE_EPOCHS}, "
    f"rank_weight={TARGET_AUC_RANK_WEIGHT}, "
    f"lr_backbone={BACKBONE_LR}, lr_head={HEAD_LR}, wd={WEIGHT_DECAY}"
)
print("=" * 110)

def sha256_file(path, block_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(block_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def stable_hash(obj):
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True, default=str).encode("utf-8")
    ).hexdigest()

CHEXZERO_SHA256 = sha256_file(CHEXZERO_CKPT) if NEEDS_CHEXZERO else "not_used"

try:
    if clip is not None:
        CLIP_MODULE_PATH = str(Path(clip.__file__).resolve())
        CLIP_MODULE_SHA256 = sha256_file(CLIP_MODULE_PATH)
    else:
        CLIP_MODULE_PATH = "not_used"
        CLIP_MODULE_SHA256 = "not_used"
except Exception:
    CLIP_MODULE_PATH = str(getattr(clip, "__file__", "unknown"))
    CLIP_MODULE_SHA256 = "unknown"

try:
    freeze = subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
    )
    (OUT_DIR / "pip_freeze.txt").write_text(freeze, encoding="utf-8")
except Exception as e:
    print("pip freeze warning:", e)

# 3. PROMPTS
# -----------------------------------------
ZS_POS = {
    "Atelectasis": [
        "atelectasis on chest x-ray",
        "lung collapse on chest radiograph",
        "plate-like atelectasis in the lung",
        "subsegmental atelectasis chest x-ray",
    ],
    "Cardiomegaly": [
        "cardiomegaly on chest x-ray",
        "enlarged cardiac silhouette radiograph",
        "cardiac enlargement chest radiograph",
        "increased cardiothoracic ratio on x-ray",
    ],
    "Consolidation": [
        "consolidation on chest x-ray",
        "airspace opacity in the lung",
        "lobar consolidation on radiograph",
        "air bronchogram in consolidated lung",
    ],
    "Edema": [
        "pulmonary edema on chest x-ray",
        "bilateral interstitial edema radiograph",
        "vascular congestion in both lungs",
        "perihilar edema on chest radiograph",
    ],
    "Pleural Effusion": [
        "pleural effusion on chest x-ray",
        "blunting of costophrenic angle",
        "pleural fluid on chest radiograph",
        "layering pleural effusion x-ray",
    ],
}

ZS_NEG = {
    "Atelectasis": [
        "no atelectasis on chest x-ray",
        "no lung collapse",
        "lungs are fully expanded",
        "no volume loss in the lung",
    ],
    "Cardiomegaly": [
        "normal heart size on chest x-ray",
        "no cardiomegaly",
        "normal cardiac silhouette",
        "normal cardiothoracic ratio",
    ],
    "Consolidation": [
        "no consolidation on chest x-ray",
        "clear lungs without airspace opacity",
        "no lobar consolidation",
        "no air bronchogram",
    ],
    "Edema": [
        "no pulmonary edema on chest x-ray",
        "no interstitial edema",
        "no vascular congestion",
        "clear lungs without edema",
    ],
    "Pleural Effusion": [
        "no pleural effusion on chest x-ray",
        "sharp costophrenic angles",
        "no pleural fluid",
        "no layering effusion",
    ],
}

CLASS_TERMS = {
    0: ["atelectasis", "collapse", "volume loss"],
    1: ["cardiomegaly", "cardiac enlargement", "enlarged heart", "cardiac silhouette"],
    2: ["consolidation", "airspace opacity", "air bronchogram"],
    3: ["edema", "vascular congestion", "interstitial opacity", "perihilar"],
    4: ["pleural effusion", "effusion", "pleural fluid", "costophrenic"],
}

NEGATION_PATTERN = re.compile(
    r"\b(no|without|negative for|free of|absent|resolved|ruled out|rule out|not seen)\b",
    re.I,
)

LABEL_ALIASES = {
    "Atelectasis": ["Atelectasis"],
    "Cardiomegaly": ["Cardiomegaly"],
    "Consolidation": ["Consolidation"],
    "Edema": ["Edema"],
    "Pleural Effusion": ["Pleural Effusion", "Pleural_Effusion", "Effusion"],
}


# -----------------------------------------
def detect_separator(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        line = f.readline()
    return "\t" if "\t" in line and line.count("\t") >= line.count(",") else ","

def read_table(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return pd.read_csv(path, sep=detect_separator(path))

def find_column(df, candidates):
    lower = {str(c).lower(): c for c in df.columns}

    for candidate in candidates:
        if candidate in df.columns:
            return candidate
        if candidate.lower() in lower:
            return lower[candidate.lower()]

    for column in df.columns:
        lc = str(column).lower()
        for candidate in candidates:
            if candidate.lower() in lc:
                return column

    return None

_IMAGE_INDEX = None

def build_image_index():
    global _IMAGE_INDEX

    if _IMAGE_INDEX is not None:
        return _IMAGE_INDEX

    print("Building NIH image index...")
    _IMAGE_INDEX = {}

    for root, _, files in os.walk(NIH_IMAGE_DIR):
        for filename in files:
            if filename.lower().endswith((".png", ".jpg", ".jpeg")):
                _IMAGE_INDEX.setdefault(filename, os.path.join(root, filename))

    print("Indexed NIH images:", len(_IMAGE_INDEX))

    if len(_IMAGE_INDEX) == 0:
        raise RuntimeError(
            f"NIH image directory exists but no PNG/JPG/JPEG images were indexed under: {NIH_IMAGE_DIR}. "
            "Check that the Kaggle dataset is fully attached and that this directory contains the image folders."
        )

    return _IMAGE_INDEX

def resolve_image_path(value):
    if value is None or pd.isna(value):
        return None

    value = str(value).strip().strip('"').strip("'")

    candidates = [value]

    if not os.path.isabs(value):
        candidates.extend([
            os.path.join(NIH_IMAGE_DIR, value),
            os.path.join(NIH_IMAGE_DIR, os.path.basename(value)),
        ])

    for path in candidates:
        if os.path.isfile(path):
            return path

    return build_image_index().get(os.path.basename(value))

def resolve_report_path(value, image_name, split):
    if value is not None and not pd.isna(value):
        value = str(value).strip().strip('"').strip("'")

        candidates = [
            value,
            os.path.join(REPORT_DIR, value),
            os.path.join(REPORT_DIR, split, os.path.basename(value)),
        ]

        for path in candidates:
            if path and os.path.isfile(path):
                return path

    stem = os.path.splitext(os.path.basename(image_name))[0]

    split_dir = {
        "train": TRAIN_REPORT_DIR,
        "val": VAL_REPORT_DIR,
        "test": TEST_REPORT_DIR,
    }[split]

    for ext in [".txt", ".report", ""]:
        path = os.path.join(split_dir, stem + ext)
        if os.path.isfile(path):
            return path

    return None

def derive_nih_patient_id(image_name):
    # Standard NIH ChestX-ray14 filename form: 00000001_000.png
    m = re.match(r"^(\d{8})_", os.path.basename(str(image_name)))
    return m.group(1) if m else ""



# -----------------------------------------

# -----------------------------------------
def get_label_matrix_and_mask(df):
    """Return Y and known-mask M with shape [N,5].

    If explicit class columns exist:
      1 = positive, 0 = negative, NaN/-1/other = unknown (masked).

    If NIH Finding Labels strings are used:
      presence = positive; absence of each target = known negative.
    """
    lower = {str(c).lower(): c for c in df.columns}
    found = {}

    for pathology in TARGET_CLASSES:
        found[pathology] = None
        for alias in LABEL_ALIASES[pathology]:
            for candidate in [alias, alias.replace(" ", "_")]:
                if candidate in df.columns:
                    found[pathology] = candidate
                    break
                if candidate.lower() in lower:
                    found[pathology] = lower[candidate.lower()]
                    break
            if found[pathology] is not None:
                break

    if all(found[p] is not None for p in TARGET_CLASSES):
        ys, ms = [], []
        for pathology in TARGET_CLASSES:
            v = pd.to_numeric(df[found[pathology]], errors="coerce").to_numpy(float)
            known = np.isfinite(v) & np.isin(v, [0.0, 1.0])
            y = np.zeros(len(v), dtype=np.float32)
            y[known] = (v[known] == 1.0).astype(np.float32)
            ys.append(y)
            ms.append(known.astype(np.float32))
        return np.column_stack(ys), np.column_stack(ms)

    label_col = find_column(
        df, ["Finding Labels", "finding_labels", "Finding_Labels", "labels"]
    )
    if label_col is None:
        raise RuntimeError(f"Could not locate NIH labels. Columns: {list(df.columns)}")

    Y = np.zeros((len(df), N_CLASSES), dtype=np.float32)
    M = np.ones((len(df), N_CLASSES), dtype=np.float32)

    for i, raw in enumerate(df[label_col].fillna("").astype(str)):
        labels = {x.strip().lower() for x in raw.split("|") if x.strip()}
        for j, pathology in enumerate(TARGET_CLASSES):
            aliases = {a.lower() for a in LABEL_ALIASES[pathology]}
            if labels & aliases:
                Y[i, j] = 1.0

    return Y, M

def label_combo_key(y, m):
    return "".join(
        "u" if mi < 0.5 else ("1" if yi >= 0.5 else "0")
        for yi, mi in zip(y, m)
    )

def standardize_split(path, split):
    df = read_table(path).copy()
    Y, M = get_label_matrix_and_mask(df)

    for j in range(N_CLASSES):
        df[Y_COLS[j]] = Y[:, j]
        df[M_COLS[j]] = M[:, j]

    df["_label_combo"] = [
        label_combo_key(Y[i], M[i]) for i in range(len(df))
    ]
    df["_target_positive_count"] = (Y * M).sum(axis=1).astype(int)

    image_col = find_column(
        df,
        [
            "image_path", "Image Path", "path", "Image Index",
            "image_index", "image_name", "image",
        ],
    )
    report_col = find_column(
        df, ["report_path", "Report Path", "report", "txt_path"]
    )
    patient_col = find_column(
        df, ["patient_id", "Patient ID", "subject_id", "patient"]
    )

    if image_col is None:
        raise RuntimeError(f"Could not identify image column. Columns: {list(df.columns)}")

    image_paths, image_names, report_paths, patient_ids = [], [], [], []

    for _, row in tqdm(
        df.iterrows(), total=len(df), desc=f"Resolve {split}"
    ):
        image_path = resolve_image_path(row[image_col])
        image_name = (
            os.path.basename(str(row[image_col]))
            if image_path is None
            else os.path.basename(image_path)
        )

        report_value = row[report_col] if report_col is not None else None
        report_path = resolve_report_path(report_value, image_name, split)

        patient_id = ""
        if patient_col is not None and not pd.isna(row[patient_col]):
            patient_id = str(row[patient_col]).strip()
        if not patient_id:
            patient_id = derive_nih_patient_id(image_name)

        image_paths.append(image_path)
        image_names.append(image_name)
        report_paths.append(report_path)
        patient_ids.append(patient_id)

    df["_image_path"] = image_paths
    df["_image_name"] = image_names
    df["_report_path"] = report_paths
    df["_patient_id"] = patient_ids

    df = df[df["_image_path"].notna()].reset_index(drop=True)

    if (df["_patient_id"].astype(str).str.len() == 0).any():
        raise RuntimeError(f"{split}: missing patient IDs")

    Y2 = df[Y_COLS].to_numpy(float)
    M2 = df[M_COLS].to_numpy(float)

    multi = int(((Y2 * M2).sum(axis=1) > 1).sum())
    allneg = int(((Y2 * M2).sum(axis=1) == 0).sum())

    print(
        f"{split}: {len(df):,} multilabel eligible | "
        f"multi-positive={multi:,} | all-five-negative={allneg:,} | "
        f"reports={df['_report_path'].notna().sum():,} | "
        f"patients={df['_patient_id'].nunique():,}"
    )
    return df

def repair_source_patient_splits(train_raw, val_raw, test_raw):
    """Preserve test first, then validation; remove overlaps from earlier pools."""
    test_patients = set(test_raw["_patient_id"].astype(str))

    train_raw = train_raw[
        ~train_raw["_patient_id"].astype(str).isin(test_patients)
    ].reset_index(drop=True)

    val_raw = val_raw[
        ~val_raw["_patient_id"].astype(str).isin(test_patients)
    ].reset_index(drop=True)

    val_patients = set(val_raw["_patient_id"].astype(str))

    train_raw = train_raw[
        ~train_raw["_patient_id"].astype(str).isin(val_patients)
    ].reset_index(drop=True)

    return train_raw, val_raw, test_raw

def combo_proportional_limit(df, n, seed):
    """Proportional sampling over joint multilabel/unknown combinations."""
    if n is None or n >= len(df):
        return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

    rng = np.random.RandomState(seed)
    groups = {
        k: np.asarray(v, dtype=int)
        for k, v in df.groupby("_label_combo", sort=True).indices.items()
    }

    ideal = {k: n * len(v) / len(df) for k, v in groups.items()}
    quota = {
        k: min(len(groups[k]), int(math.floor(ideal[k])))
        for k in groups
    }

    used = sum(quota.values())
    order = sorted(
        groups,
        key=lambda k: (
            ideal[k] - math.floor(ideal[k]),
            len(groups[k]),
            k,
        ),
        reverse=True,
    )

    while used < n:
        progressed = False
        for k in order:
            if quota[k] < len(groups[k]):
                quota[k] += 1
                used += 1
                progressed = True
            if used == n:
                break
        if not progressed:
            break

    selected = []
    for k in sorted(groups):
        if quota[k]:
            selected.extend(
                rng.choice(groups[k], size=quota[k], replace=False).tolist()
            )

    if len(selected) != n:
        raise RuntimeError(
            f"multilabel sampler selected {len(selected)} rows; expected {n}"
        )

    return (
        df.iloc[selected]
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )

def assert_patient_disjoint(a_name, a_df, b_name, b_df):
    overlap = (
        set(a_df["_patient_id"].astype(str))
        & set(b_df["_patient_id"].astype(str))
    )
    if overlap:
        raise RuntimeError(
            f"PATIENT LEAKAGE {a_name}<->{b_name}: {len(overlap)} patients"
        )

def assert_multilabel_integrity(name, df):
    Y = df[Y_COLS].to_numpy(float)
    M = df[M_COLS].to_numpy(float)

    for c, pathology in enumerate(TARGET_CLASSES):
        known = M[:, c] > 0.5
        values = Y[known, c]
        if known.sum() < 2 or len(np.unique(values)) < 2:
            raise RuntimeError(
                f"{name}/{pathology}: needs both positive and negative known labels"
            )

    print(
        f"{name} integrity: OK | images={len(df):,} | "
        f"patients={df['_patient_id'].nunique():,}"
    )

raw_train_df = standardize_split(TRAIN_CSV, "train")
raw_val_df = standardize_split(VAL_CSV, "val")
raw_test_df = standardize_split(TEST_CSV, "test")

# Fail loudly if the supplied CSVs were already reduced to a single-label task upstream.
# A genuine multilabel evaluation cannot reconstruct rows that were removed before this notebook.
for _split_name, _split_df in [
    ("train", raw_train_df),
    ("val", raw_val_df),
    ("test", raw_test_df),
]:
    _Y = _split_df[Y_COLS].to_numpy(float)
    _M = _split_df[M_COLS].to_numpy(float)
    _multi_positive = int(((_Y * _M).sum(axis=1) > 1).sum())
    if _multi_positive == 0:
        raise RuntimeError(
            f"{_split_name}: zero multi-positive cases were found in the supplied split. "
            "The input CSV appears to have been prefiltered to a single-label task, so this "
            "notebook cannot honestly claim genuine multilabel evaluation. Use the original "
            "NIH labels/splits (or regenerate these split files without removing multi-positive rows)."
        )

raw_train_df, raw_val_df, raw_test_df = repair_source_patient_splits(
    raw_train_df, raw_val_df, raw_test_df
)

# P0 uses reports only for TRAIN-TIME weak patch mining.
train_pool = raw_train_df[
    raw_train_df["_report_path"].apply(
        lambda x: isinstance(x, str) and os.path.isfile(x)
    )
].reset_index(drop=True)

if len(train_pool) == 0:
    raise RuntimeError("No training images with valid reports were found for P0")

train_df = combo_proportional_limit(
    train_pool, MAX_TRAIN_IMAGES, SEED_EXTRACT
)
val_df = combo_proportional_limit(
    raw_val_df, MAX_VAL_IMAGES, SEED_EXTRACT + 1
)
test_df = combo_proportional_limit(
    raw_test_df, MAX_TEST_IMAGES, SEED_EXTRACT + 2
)

assert_patient_disjoint("train", train_df, "val", val_df)
assert_patient_disjoint("train", train_df, "test", test_df)
assert_patient_disjoint("val", val_df, "test", test_df)

for split_name, df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df),
]:
    assert_multilabel_integrity(split_name, df)
    df.to_csv(OUT_DIR / f"cohort_{split_name}.csv", index=False)

    Y = df[Y_COLS].to_numpy(float)
    M = df[M_COLS].to_numpy(float)

    print(f"\n{split_name.upper()} CLASS COUNTS")
    for c, pathology in enumerate(TARGET_CLASSES):
        known = M[:, c] > 0.5
        print(
            f"{pathology:18s} | "
            f"pos={int(Y[known, c].sum()):4d} | "
            f"known={int(known.sum()):4d} | "
            f"prev={Y[known, c].mean():.4f}"
        )

COHORT_HASH = stable_hash({
    "train": train_df[
        ["_image_name", "_patient_id", "_report_path"] + Y_COLS + M_COLS
    ].astype(str).values.tolist(),
    "val": val_df[
        ["_image_name", "_patient_id"] + Y_COLS + M_COLS
    ].astype(str).values.tolist(),
    "test": test_df[
        ["_image_name", "_patient_id"] + Y_COLS + M_COLS
    ].astype(str).values.tolist(),
})

def cohort_label_payload(row):
    payload = {}
    for j in range(N_CLASSES):
        payload[PATCH_Y_COLS[j]] = float(row[Y_COLS[j]])
        payload[PATCH_M_COLS[j]] = float(row[M_COLS[j]])
    return payload

def patch_label_payload(row):
    payload = {}
    for j in range(N_CLASSES):
        payload[PATCH_Y_COLS[j]] = float(row[PATCH_Y_COLS[j]])
        payload[PATCH_M_COLS[j]] = float(row[PATCH_M_COLS[j]])
    return payload

# 7. CHEXZERO CHECKPOINT LOADING

# -----------------------------------------
def clean_state_dict(checkpoint_path):
    state = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    if isinstance(state, dict):
        for key in ("state_dict", "model", "model_state_dict", "net"):
            if key in state and isinstance(state[key], dict):
                state = state[key]
                break

    clean = {}

    for key, value in state.items():
        new_key = key

        for prefix in ("module.", "model."):
            if new_key.startswith(prefix):
                new_key = new_key[len(prefix):]
                break

        clean[new_key] = value

    return clean

def load_chexzero():
    # Your supplied checkpoint is loaded against ViT-B/32, matching the original notebook.
    model, preprocess = clip.load(
        "ViT-B/32",
        device=DEVICE,
        jit=False,
    )

    clean = clean_state_dict(CHEXZERO_CKPT)

    incompatible = model.load_state_dict(
        clean,
        strict=False,
    )

    missing = list(incompatible.missing_keys)
    unexpected = list(incompatible.unexpected_keys)

    print(
        f"CheXzero load: missing={len(missing)} | unexpected={len(unexpected)}"
    )

    if missing:
        print("Missing:", missing)

    if unexpected:
        print("Unexpected:", unexpected)

    if (
        len(missing) > CKPT_MAX_MISSING
        or len(unexpected) > CKPT_MAX_UNEXPECTED
    ):
        raise RuntimeError(
            "CheXzero checkpoint incompatibility exceeds the declared budget. "
            "Inspect the exact checkpoint/model architecture before proceeding."
        )

    model = model.float().eval()

    for parameter in model.parameters():
        parameter.requires_grad_(False)

    return model, preprocess

@torch.no_grad()
def encode_texts(model, texts):
    tokens = clip.tokenize(
        list(texts),
        truncate=True,
    ).to(DEVICE)

    embeddings = model.encode_text(tokens).float()

    return F.normalize(
        embeddings,
        dim=-1,
    )

@torch.no_grad()
def build_prompt_matrix(model, prompt_dictionary):
    vectors = []

    for pathology in TARGET_CLASSES:
        embeddings = encode_texts(
            model,
            prompt_dictionary[pathology],
        )

        mean_vector = F.normalize(
            embeddings.mean(dim=0),
            dim=0,
        )

        vectors.append(mean_vector)

    return torch.stack(vectors, dim=0)

# -----------------------------------------

# -----------------------------------------
def read_report(path):
    if not path or not os.path.isfile(path):
        return ""

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="ignore",
    ) as f:
        return f.read()

def get_report_queries(report_text):
    sentences = [
        re.sub(r"\s+", " ", sentence).strip()
        for sentence in re.split(
            r"(?<=[.!?])\s+|\n+",
            report_text,
        )
        if len(sentence.strip()) > 3
    ]

    selected_sentences = []
    active_classes = []

    for sentence in sentences:
        lower = sentence.lower()

        for class_id, terms in CLASS_TERMS.items():
            contains_target_term = any(
                term in lower
                for term in terms
            )

            if (
                contains_target_term
                and not NEGATION_PATTERN.search(lower)
            ):
                selected_sentences.append(sentence[:350])
                active_classes.append(class_id)
                break

    # Never use the ground-truth label as fallback.
    # If no explicit positive sentence is found, use generic fixed prompts.
    if not selected_sentences:
        return [], list(range(N_CLASSES))

    return (
        selected_sentences[:6],
        sorted(set(active_classes)),
    )

# -----------------------------------------

# -----------------------------------------
def build_candidate_grid():
    boxes = []

    for scale in PATCH_SCALES:
        stride = max(
            16,
            int(scale * STRIDE_FRAC),
        )

        for y in range(
            0,
            WORK_IMAGE_SIZE - scale + 1,
            stride,
        ):
            for x in range(
                0,
                WORK_IMAGE_SIZE - scale + 1,
                stride,
            ):
                boxes.append(
                    (x, y, x + scale, y + scale)
                )

    boxes = list(dict.fromkeys(boxes))

    if len(boxes) > MAX_CANDIDATES_PER_IMAGE:
        indices = np.linspace(
            0,
            len(boxes) - 1,
            MAX_CANDIDATES_PER_IMAGE,
        ).round().astype(int)

        boxes = [boxes[i] for i in indices]

    return boxes

TRAIN_GRID_BOXES = build_candidate_grid()

def fixed_eval_boxes():
    # Five deterministic anchors per scale:
    # TL, TR, BL, BR, center.
    boxes = []

    for scale in EVAL_PATCH_SCALES:
        m = WORK_IMAGE_SIZE - scale
        center = m // 2

        coords = [
            (0, 0),
            (m, 0),
            (0, m),
            (m, m),
            (center, center),
        ]

        for x, y in coords:
            boxes.append(
                (x, y, x + scale, y + scale)
            )

    return list(dict.fromkeys(boxes))

EVAL_BOXES = fixed_eval_boxes()

print(
    "Train candidate boxes/image:",
    len(TRAIN_GRID_BOXES),
)

print(
    "Fixed report-free eval boxes/image:",
    len(EVAL_BOXES),
)

# -----------------------------------------

# -----------------------------------------
@torch.no_grad()
def encode_patch_candidates(
    model,
    preprocess,
    image_512,
):
    outputs = []

    for start in range(
        0,
        len(TRAIN_GRID_BOXES),
        PATCH_ENCODE_BATCH,
    ):
        boxes = TRAIN_GRID_BOXES[
            start:
            start + PATCH_ENCODE_BATCH
        ]

        batch = torch.stack([
            preprocess(
                image_512.crop(box)
            )
            for box in boxes
        ]).to(DEVICE)

        amp_ctx = (
            torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            )
            if USE_CUDA
            else nullcontext()
        )

        with amp_ctx:
            embeddings = model.encode_image(batch).float()

        embeddings = F.normalize(
            embeddings,
            dim=-1,
        )

        outputs.append(
            embeddings.cpu()
        )

    return torch.cat(
        outputs,
        dim=0,
    ).to(DEVICE)

class ViTGradCAM:
    def __init__(self, model, preprocess):
        self.model = model
        self.preprocess = preprocess
        self.target_layer = (
            model.visual.transformer.resblocks[-1].ln_1
        )

    def __call__(
        self,
        image_512,
        target_text_embedding,
    ):
        cache = {}

        def hook(module, inputs, output):
            cache["activation"] = output

        handle = self.target_layer.register_forward_hook(
            hook
        )

        try:
            with torch.enable_grad():
                image_tensor = (
                    self.preprocess(image_512)
                    .unsqueeze(0)
                    .to(DEVICE)
                    .float()
                )

                image_tensor.requires_grad_(True)

                image_embedding = F.normalize(
                    self.model.encode_image(
                        image_tensor
                    ).float(),
                    dim=-1,
                )

                target = F.normalize(
                    target_text_embedding.float(),
                    dim=-1,
                ).unsqueeze(0)

                score = (
                    image_embedding
                    * target
                ).sum()

                activations = cache["activation"]

                gradients = torch.autograd.grad(
                    score,
                    activations,
                    retain_graph=False,
                    create_graph=False,
                )[0]

                # OpenAI CLIP transformer layout: [sequence, batch, channels]
                if (
                    activations.ndim == 3
                    and activations.shape[1] == 1
                ):
                    activations = activations[:, 0, :]
                    gradients = gradients[:, 0, :]
                else:
                    activations = activations[0]
                    gradients = gradients[0]

                # Transformer Grad-CAM: compute one channel weight per embedding
                # dimension by spatially averaging gradients over PATCH tokens,
                # then form the CAM as ReLU(sum_k alpha_k * A_k).
                # This is conventional Grad-CAM adapted to ViT token activations;
                # it is deliberately NOT token-wise gradient*activation saliency.
                patch_activations = activations[1:]
                patch_gradients = gradients[1:]

                channel_weights = patch_gradients.mean(
                    dim=0
                )

                token_importance = F.relu(
                    (
                        patch_activations
                        * channel_weights.unsqueeze(0)
                    ).sum(dim=-1)
                )

                side = int(
                    math.sqrt(
                        token_importance.numel()
                    )
                )

                token_importance = token_importance[
                    : side * side
                ]

                heatmap = token_importance.view(
                    1,
                    1,
                    side,
                    side,
                )

                heatmap = F.interpolate(
                    heatmap,
                    size=(
                        WORK_IMAGE_SIZE,
                        WORK_IMAGE_SIZE,
                    ),
                    mode="bilinear",
                    align_corners=False,
                )[0, 0]

                heatmap = heatmap - heatmap.min()

                heatmap = heatmap / (
                    heatmap.max() + 1e-8
                )

                return (
                    heatmap
                    .detach()
                    .cpu()
                )

        finally:
            handle.remove()

            self.model.zero_grad(
                set_to_none=True
            )

def gradcam_scores_for_train_boxes(heatmap):
    scores = []

    for x1, y1, x2, y2 in TRAIN_GRID_BOXES:
        region = heatmap[y1:y2, x1:x2]

        scores.append(
            float(region.mean())
            if region.numel()
            else 0.0
        )

    return torch.tensor(
        scores,
        dtype=torch.float32,
        device=DEVICE,
    )

def binary_positive_probability(
    patch_embeddings,
    positive_matrix,
    negative_matrix,
):
    pos_similarity = (
        patch_embeddings
        @ positive_matrix.T
    )

    neg_similarity = (
        patch_embeddings
        @ negative_matrix.T
    )

    logits = torch.stack(
        [
            pos_similarity,
            neg_similarity,
        ],
        dim=-1,
    )

    logits = logits * BINARY_PROB_TEMP

    return torch.softmax(
        logits,
        dim=-1,
    )[..., 0]

def combine_scores(
    semantic,
    probability,
    gradcam,
    use_semantic,
    use_probability,
    use_gradcam,
):
    components = []
    weights = []
    eps = 1e-6

    if use_semantic:
        components.append(
            semantic.clamp(eps, 1.0)
        )
        weights.append(W_SEMANTIC)

    if use_probability:
        components.append(
            probability.clamp(eps, 1.0)
        )
        weights.append(W_PROBABILITY)

    if use_gradcam:
        components.append(
            gradcam.clamp(eps, 1.0)
        )
        weights.append(W_GRADCAM)

    if not components:
        raise RuntimeError(
            "At least one patch-selection signal must be active."
        )

    total_weight = sum(weights)

    combined = torch.zeros_like(
        components[0]
    )

    for component, weight in zip(
        components,
        weights,
    ):
        combined += (
            weight / total_weight
        ) * torch.log(component)

    return torch.exp(combined)

def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    intersection = (
        max(0, ix2 - ix1)
        * max(0, iy2 - iy1)
    )

    area_a = (
        ax2 - ax1
    ) * (
        ay2 - ay1
    )

    area_b = (
        bx2 - bx1
    ) * (
        by2 - by1
    )

    return intersection / (
        area_a
        + area_b
        - intersection
        + 1e-9
    )

def select_top_k_nonredundant(scores):
    ordering = torch.argsort(
        scores,
        descending=True,
    ).tolist()

    selected = []

    for idx in ordering:
        if all(
            box_iou(
                TRAIN_GRID_BOXES[idx],
                TRAIN_GRID_BOXES[prev],
            ) < NMS_IOU_THRESHOLD
            for prev in selected
        ):
            selected.append(idx)

        if len(selected) >= TOPK_PATCHES_PER_IMAGE:
            break

    # Preserve exactly K patches per image when enough candidates exist.
    if len(selected) < TOPK_PATCHES_PER_IMAGE:
        for idx in ordering:
            if idx not in selected:
                selected.append(idx)

            if len(selected) >= TOPK_PATCHES_PER_IMAGE:
                break

    return selected

# -----------------------------------------


def deterministic_uint32(*parts):
    payload = "\x1f".join(map(str, parts)).encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "little", signed=False)

def random_box_same_scale(scale, rng, forbidden_boxes):
    candidates = [
        box for box in TRAIN_GRID_BOXES
        if (box[2] - box[0]) == int(scale)
    ]
    if not candidates:
        raise RuntimeError(f"No candidate boxes exist for scale={scale}")

    order = rng.permutation(len(candidates)).tolist()

    for idx in order:
        candidate = candidates[idx]
        if all(
            box_iou(candidate, forbidden) < NMS_IOU_THRESHOLD
            for forbidden in forbidden_boxes
        ):
            return candidate

    def overlap_penalty(candidate):
        if not forbidden_boxes:
            return 0.0
        return max(
            box_iou(candidate, forbidden)
            for forbidden in forbidden_boxes
        )

    return min(candidates, key=overlap_penalty)


# -------------------------------------------------------------------------------------------------
# ARM-SPECIFIC TRAIN PATCH MINING + COMMON REPORT-FREE VAL/TEST MINING
# -------------------------------------------------------------------------------------------------
EXTRACTION_CONFIG = {
    "schema": EXPERIMENT_SCHEMA,
    "arm": ARM_NAME,
    "arm_config": ARM_CONFIG,
    "cohort_hash": COHORT_HASH,
    "chexzero_sha256": CHEXZERO_SHA256,
    "topk": TOPK_PATCHES_PER_IMAGE,
    "grid": TRAIN_GRID_BOXES,
    "weights": {
        "semantic": W_SEMANTIC,
        "probability": W_PROBABILITY,
        "gradcam": W_GRADCAM,
    },
}
EXTRACTION_HASH = stable_hash(EXTRACTION_CONFIG)[:20]

(OUT_DIR / "extraction_config.json").write_text(
    json.dumps(EXTRACTION_CONFIG, indent=2, default=str),
    encoding="utf-8",
)

def atomic_pickle(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "wb") as f:
        pickle.dump(obj, f, pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)

def _append_patch_record(records, row, image_name, rank, box, score=np.nan):
    x1, y1, x2, y2 = box
    records.append({
        "variant": ARM_NAME,
        "image_name": image_name,
        "patient_id": str(row["_patient_id"]),
        "image_path": row["_image_path"],
        "x1": int(x1),
        "y1": int(y1),
        "x2": int(x2),
        "y2": int(y2),
        "scale": int(x2 - x1),
        "rank": int(rank),
        "selection_score": float(score) if np.isfinite(score) else np.nan,
        **cohort_label_payload(row),
    })


def validate_patch_table_geometry(table, table_name):
    """
    Strict pre-training geometry audit.

    The old v2 code had patch-label columns y1/y2 that collided with crop-coordinate
    columns y1/y2. V3 uses collision-proof target_*/known_* labels and verifies every
    patch before training.
    """
    coordinate_columns = ["x1", "y1", "x2", "y2"]
    required = (
        ["image_name", "patient_id", "image_path", "rank", "scale"]
        + coordinate_columns
        + PATCH_Y_COLS
        + PATCH_M_COLS
    )

    missing = [c for c in required if c not in table.columns]
    if missing:
        raise RuntimeError(
            f"{table_name}: missing required patch-table columns: {missing}"
        )

    reserved_geometry = {
        "x1", "y1", "x2", "y2", "scale", "rank",
        "image_name", "patient_id", "image_path",
    }
    label_namespace = set(PATCH_Y_COLS) | set(PATCH_M_COLS)
    collision = sorted(reserved_geometry & label_namespace)
    if collision:
        raise RuntimeError(
            f"{table_name}: label/geometry namespace collision: {collision}"
        )

    numeric = table[coordinate_columns + ["scale", "rank"]].apply(
        pd.to_numeric,
        errors="coerce",
    )

    values = numeric[coordinate_columns + ["scale"]].to_numpy(dtype=np.float64)
    nonfinite = ~np.isfinite(values).all(axis=1)

    x1 = numeric["x1"].to_numpy(dtype=np.float64)
    y1 = numeric["y1"].to_numpy(dtype=np.float64)
    x2 = numeric["x2"].to_numpy(dtype=np.float64)
    y2 = numeric["y2"].to_numpy(dtype=np.float64)
    scale = numeric["scale"].to_numpy(dtype=np.float64)

    invalid = (
        nonfinite
        | (x1 < 0)
        | (y1 < 0)
        | (x2 > WORK_IMAGE_SIZE)
        | (y2 > WORK_IMAGE_SIZE)
        | (x2 <= x1)
        | (y2 <= y1)
        | (scale <= 0)
        | (np.abs((x2 - x1) - scale) > 1e-6)
        | (np.abs((y2 - y1) - scale) > 1e-6)
    )

    if invalid.any():
        bad_idx = np.flatnonzero(invalid)[:10]
        examples = table.iloc[bad_idx][
            ["image_name", "rank", "x1", "y1", "x2", "y2", "scale"]
        ].to_dict("records")
        raise RuntimeError(
            f"{table_name}: {int(invalid.sum())} invalid crop boxes detected BEFORE training. "
            f"Examples: {examples}"
        )

    labels = table[PATCH_Y_COLS].apply(
        pd.to_numeric, errors="coerce"
    ).to_numpy(dtype=np.float64)
    masks = table[PATCH_M_COLS].apply(
        pd.to_numeric, errors="coerce"
    ).to_numpy(dtype=np.float64)

    if not np.isfinite(labels).all():
        raise RuntimeError(f"{table_name}: non-finite target values")
    if not np.isfinite(masks).all():
        raise RuntimeError(f"{table_name}: non-finite known-mask values")
    if not np.isin(labels, [0.0, 1.0]).all():
        raise RuntimeError(f"{table_name}: targets must be binary 0/1")
    if not np.isin(masks, [0.0, 1.0]).all():
        raise RuntimeError(f"{table_name}: known masks must be binary 0/1")

    counts = table.groupby("image_name", sort=False).size()
    if len(counts) and not (counts == TOPK_PATCHES_PER_IMAGE).all():
        bad = counts[counts != TOPK_PATCHES_PER_IMAGE].head(10).to_dict()
        raise RuntimeError(
            f"{table_name}: expected {TOPK_PATCHES_PER_IMAGE} patches/image; "
            f"bad examples: {bad}"
        )

    print(
        f"{table_name} geometry audit: OK | "
        f"patches={len(table):,} | images={table['image_name'].nunique():,} | "
        f"K={TOPK_PATCHES_PER_IMAGE}"
    )
    return table


def validated_crop_box(row, image_name, rank):
    try:
        x1 = int(row["x1"])
        y1 = int(row["y1"])
        x2 = int(row["x2"])
        y2 = int(row["y2"])
    except Exception as exc:
        raise RuntimeError(
            f"{image_name} rank={rank}: non-integer crop coordinates"
        ) from exc

    if not (
        0 <= x1 < x2 <= WORK_IMAGE_SIZE
        and 0 <= y1 < y2 <= WORK_IMAGE_SIZE
    ):
        raise RuntimeError(
            f"{image_name} rank={rank}: invalid crop box "
            f"(x1={x1}, y1={y1}, x2={x2}, y2={y2})"
        )

    return (x1, y1, x2, y2)


def extract_train_patches_for_arm(
    dataframe,
    model,
    preprocess,
    positive_matrix,
    negative_matrix,
):
    cache_path = OUT_DIR / f"{ARM_NAME}_train_{EXTRACTION_HASH}.pkl"
    records = []
    completed = set()

    if cache_path.exists():
        with open(cache_path, "rb") as f:
            cache = pickle.load(f)
        if cache.get("hash") == EXTRACTION_HASH:
            records = cache.get("records", [])
            completed = set(cache.get("completed", []))
            print(
                f"[resume] {ARM_NAME} train mining: "
                f"{len(completed):,}/{len(dataframe):,} images"
            )

    cam_engine = ViTGradCAM(model, preprocess)

    for _, row in tqdm(
        dataframe.iterrows(),
        total=len(dataframe),
        desc=f"{ARM_NAME} train patch mining",
    ):
        image_name = str(row["_image_name"])
        if image_name in completed:
            continue

        image = (
            Image.open(row["_image_path"])
            .convert("RGB")
            .resize(
                (WORK_IMAGE_SIZE, WORK_IMAGE_SIZE),
                Image.BICUBIC,
            )
        )

        patch_embeddings = encode_patch_candidates(
            model, preprocess, image
        )

        all_class_probabilities = binary_positive_probability(
            patch_embeddings,
            positive_matrix,
            negative_matrix,
        )

        # Report-free fixed prompt signals.
        no_report_semantic = (
            (patch_embeddings @ positive_matrix.T).max(dim=1).values + 1.0
        ) / 2.0

        no_report_probability = (
            all_class_probabilities.max(dim=1).values
        )

        no_report_target = F.normalize(
            positive_matrix.mean(dim=0), dim=0
        )

        # Report-guided signals are built ONLY for arms that are allowed to use reports.
        need_report_signals = bool(
            ARM_CONFIG["use_report"]
            or ARM_CONFIG["random_matched"]
        )

        report_semantic = None
        report_probability = None
        report_target = None

        if need_report_signals:
            report_queries, active_classes = get_report_queries(
                read_report(row["_report_path"])
            )

            if report_queries:
                report_query_matrix = encode_texts(
                    model, report_queries
                )
            else:
                report_query_matrix = positive_matrix
                active_classes = list(range(N_CLASSES))

            report_semantic = (
                (patch_embeddings @ report_query_matrix.T).max(dim=1).values + 1.0
            ) / 2.0

            report_probability = (
                all_class_probabilities[:, active_classes].max(dim=1).values
            )

            report_target = F.normalize(
                report_query_matrix.mean(dim=0), dim=0
            )

        # Select the signal family allowed for this arm.
        if ARM_CONFIG["use_report"]:
            semantic = report_semantic
            probability = report_probability
            gradcam_target = report_target
        else:
            semantic = no_report_semantic
            probability = no_report_probability
            gradcam_target = no_report_target

        if ARM_CONFIG["random_matched"]:
            # P5 is count/scale matched to the full P0 selector.
            full_gradcam = gradcam_scores_for_train_boxes(
                cam_engine(image, report_target)
            )
            full_scores = combine_scores(
                report_semantic,
                report_probability,
                full_gradcam,
                True,
                True,
                True,
            )
            full_indices = select_top_k_nonredundant(full_scores)
            full_boxes = [TRAIN_GRID_BOXES[i] for i in full_indices]

            rng = np.random.RandomState(
                deterministic_uint32(
                    SEED_EXTRACT,
                    EXTRACTION_HASH,
                    ARM_NAME,
                    image_name,
                )
            )

            random_boxes = []
            for rank, full_box in enumerate(full_boxes):
                scale = int(full_box[2] - full_box[0])
                box = random_box_same_scale(
                    scale,
                    rng,
                    random_boxes,
                )
                random_boxes.append(box)
                _append_patch_record(
                    records,
                    row,
                    image_name,
                    rank,
                    box,
                    np.nan,
                )

        else:
            if ARM_CONFIG["use_gradcam"]:
                gradcam = gradcam_scores_for_train_boxes(
                    cam_engine(image, gradcam_target)
                )
            else:
                gradcam = torch.ones(
                    len(TRAIN_GRID_BOXES),
                    dtype=torch.float32,
                    device=DEVICE,
                )

            scores = combine_scores(
                semantic,
                probability,
                gradcam,
                ARM_CONFIG["use_semantic"],
                ARM_CONFIG["use_probability"],
                ARM_CONFIG["use_gradcam"],
            )

            selected = select_top_k_nonredundant(scores)

            for rank, patch_index in enumerate(selected):
                _append_patch_record(
                    records,
                    row,
                    image_name,
                    rank,
                    TRAIN_GRID_BOXES[patch_index],
                    float(scores[patch_index].detach().cpu()),
                )

        completed.add(image_name)

        if len(completed) % 50 == 0:
            atomic_pickle(
                {
                    "hash": EXTRACTION_HASH,
                    "records": records,
                    "completed": list(completed),
                },
                cache_path,
            )

    atomic_pickle(
        {
            "hash": EXTRACTION_HASH,
            "records": records,
            "completed": list(completed),
        },
        cache_path,
    )

    table = pd.DataFrame(records)
    expected = len(dataframe) * TOPK_PATCHES_PER_IMAGE

    if len(table) != expected:
        raise RuntimeError(
            f"{ARM_NAME}: expected {expected:,} train patches, got {len(table):,}"
        )

    counts = table.groupby("image_name").size()
    if not (counts == TOPK_PATCHES_PER_IMAGE).all():
        raise RuntimeError(
            f"{ARM_NAME}: inconsistent patches/image"
        )

    table.to_csv(
        OUT_DIR / f"{ARM_NAME}_train_patches.csv",
        index=False,
    )

    return table

def build_common_image_only_eval_patches(
    dataframe,
    split_name,
    model,
    preprocess,
    positive_matrix,
    negative_matrix,
):
    # Same report-free inference miner for EVERY patch arm.
    split_signature = stable_hash(
        dataframe[
            ["_image_name", "_patient_id"] + Y_COLS + M_COLS
        ].astype(str).values.tolist()
    )

    eval_hash = stable_hash({
        "schema": EXPERIMENT_SCHEMA,
        "split": split_name,
        "split_signature": split_signature,
        "topk": TOPK_PATCHES_PER_IMAGE,
        "grid": TRAIN_GRID_BOXES,
        "weights": [W_SEMANTIC, W_PROBABILITY, W_GRADCAM],
        "mode": "common_full_image_only_miner",
    })[:16]

    cache_csv = OUT_DIR / f"{split_name}_common_image_only_{eval_hash}.csv"

    if cache_csv.exists():
        table = pd.read_csv(cache_csv)
        expected = len(dataframe) * TOPK_PATCHES_PER_IMAGE
        if len(table) == expected:
            print(f"[cache] common {split_name} eval patches")
            return table

    cam_engine = ViTGradCAM(model, preprocess)
    generic_target = F.normalize(
        positive_matrix.mean(dim=0), dim=0
    )

    records = []

    for _, row in tqdm(
        dataframe.iterrows(),
        total=len(dataframe),
        desc=f"{split_name} common image-only mining",
    ):
        image = (
            Image.open(row["_image_path"])
            .convert("RGB")
            .resize(
                (WORK_IMAGE_SIZE, WORK_IMAGE_SIZE),
                Image.BICUBIC,
            )
        )

        patch_embeddings = encode_patch_candidates(
            model, preprocess, image
        )

        semantic = (
            (patch_embeddings @ positive_matrix.T).max(dim=1).values + 1.0
        ) / 2.0

        probability = binary_positive_probability(
            patch_embeddings,
            positive_matrix,
            negative_matrix,
        ).max(dim=1).values

        gradcam = gradcam_scores_for_train_boxes(
            cam_engine(image, generic_target)
        )

        scores = combine_scores(
            semantic,
            probability,
            gradcam,
            True,
            True,
            True,
        )

        selected = select_top_k_nonredundant(scores)

        for rank, patch_index in enumerate(selected):
            x1, y1, x2, y2 = TRAIN_GRID_BOXES[patch_index]
            records.append({
                "variant": "COMMON_IMAGE_ONLY_EVAL",
                "split": split_name,
                "image_name": str(row["_image_name"]),
                "patient_id": str(row["_patient_id"]),
                "image_path": row["_image_path"],
                "x1": int(x1),
                "y1": int(y1),
                "x2": int(x2),
                "y2": int(y2),
                "scale": int(x2 - x1),
                "rank": int(rank),
                **cohort_label_payload(row),
            })

    table = pd.DataFrame(records)
    expected = len(dataframe) * TOPK_PATCHES_PER_IMAGE

    if len(table) != expected:
        raise RuntimeError(
            f"{split_name}: expected {expected:,} eval patches, got {len(table):,}"
        )

    counts = table.groupby("image_name").size()
    if not (counts == TOPK_PATCHES_PER_IMAGE).all():
        raise RuntimeError(
            f"{split_name}: inconsistent eval patches/image"
        )

    table.to_csv(cache_csv, index=False)
    return table


# -------------------------------------------------------------------------------------------------
# PIPELINE-4-LIKE 6-PATCH MIL DATASET / BACKBONE / LOSS / METRICS
# -------------------------------------------------------------------------------------------------
def pipeline4_transform(train=False):
    ops = [T.Resize((224, 224))]
    if train:
        ops += [
            T.RandomAffine(
                degrees=3,
                translate=(0.02, 0.02),
                scale=(0.97, 1.03),
            )
        ]
    ops += [
        T.ToTensor(),
        T.Normalize(
            (0.485, 0.456, 0.406),
            (0.229, 0.224, 0.225),
        ),
    ]
    return T.Compose(ops)

class PatchBagDataset(Dataset):
    def __init__(self, table, train=False):
        self.table = validate_patch_table_geometry(
            table.copy().reset_index(drop=True),
            table_name="PatchBagDataset",
        )
        self.transform = pipeline4_transform(train)
        self.groups = []

        for image_name, group in self.table.groupby("image_name", sort=True):
            group = group.sort_values("rank").reset_index(drop=True)

            if len(group) != TOPK_PATCHES_PER_IMAGE:
                raise RuntimeError(
                    f"{image_name}: expected {TOPK_PATCHES_PER_IMAGE} patches, got {len(group)}"
                )

            ranks = group["rank"].astype(int).tolist()
            if ranks != list(range(TOPK_PATCHES_PER_IMAGE)):
                raise RuntimeError(
                    f"{image_name}: ranks must be 0..{TOPK_PATCHES_PER_IMAGE-1}; got {ranks}"
                )

            # Every patch in one image must carry the same label vector/mask.
            y = group.loc[0, PATCH_Y_COLS].to_numpy(dtype=np.float32)
            m = group.loc[0, PATCH_M_COLS].to_numpy(dtype=np.float32)

            for idx in range(1, len(group)):
                if not np.array_equal(
                    group.loc[idx, PATCH_Y_COLS].to_numpy(dtype=np.float32),
                    y,
                ):
                    raise RuntimeError(f"{image_name}: inconsistent labels across patches")
                if not np.array_equal(
                    group.loc[idx, PATCH_M_COLS].to_numpy(dtype=np.float32),
                    m,
                ):
                    raise RuntimeError(f"{image_name}: inconsistent masks across patches")

            self.groups.append((image_name, group, y, m))

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, index):
        image_name, group, y, m = self.groups[index]

        image = (
            Image.open(group.loc[0, "image_path"])
            .convert("RGB")
            .resize(
                (WORK_IMAGE_SIZE, WORK_IMAGE_SIZE),
                Image.BICUBIC,
            )
        )

        crops = []
        for _, row in group.iterrows():
            rank = int(row["rank"])
            box = validated_crop_box(
                row,
                image_name=str(image_name),
                rank=rank,
            )
            patch = image.crop(box)

            if patch.width <= 0 or patch.height <= 0:
                raise RuntimeError(
                    f"{image_name} rank={rank}: PIL produced an empty patch for box={box}"
                )

            crops.append(self.transform(patch))

        return (
            torch.stack(crops),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(m, dtype=torch.float32),
            str(image_name),
            str(group.loc[0, "patient_id"]),
        )

class WholeImageBagDataset(Dataset):
    """Compute-matched whole-image control: K whole-image views per image."""
    def __init__(self, dataframe, train=False):
        self.df = dataframe.reset_index(drop=True)
        self.transform = pipeline4_transform(train)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image = Image.open(row["_image_path"]).convert("RGB")

        # At train time RandomAffine is independently sampled K times.
        # At evaluation, deterministic transforms make all K views identical,
        # and logsumexp(logits)-log(K) reduces exactly to the whole-image logit.
        views = [
            self.transform(image.copy())
            for _ in range(TOPK_PATCHES_PER_IMAGE)
        ]

        y = row[Y_COLS].to_numpy(dtype=np.float32)
        m = row[M_COLS].to_numpy(dtype=np.float32)

        return (
            torch.stack(views),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(m, dtype=torch.float32),
            str(row["_image_name"]),
            str(row["_patient_id"]),
        )

def make_loader(dataset, batch_size, shuffle, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)

    def seed_worker(worker_id):
        worker_seed = torch.initial_seed() % 2**32
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    kwargs = dict(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=generator,
    )

    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = True
        kwargs["prefetch_factor"] = 2

    return DataLoader(**kwargs)

class CleanImageNetViTEncoder(nn.Module):
    """Matches Pipeline 4 headline representation backbone."""
    def __init__(self):
        super().__init__()

        try:
            weights = ViT_B_16_Weights.IMAGENET1K_V1
            self.model = vit_b_16(weights=weights)
        except Exception as e:
            raise RuntimeError(
                "Could not load torchvision ViT-B/16 IMAGENET1K_V1 weights. "
                "Enable Kaggle Internet once or make sure the checkpoint is cached. "
                "This strict notebook refuses to silently switch backbones."
            ) from e

        self.model.heads = nn.Identity()
        self.depth = len(self.model.encoder.layers)

        if self.depth < UNFREEZE_BLOCKS:
            raise RuntimeError(
                f"ViT depth {self.depth} < requested unfreeze depth {UNFREEZE_BLOCKS}"
            )

        self.set_trainable_blocks(UNFREEZE_BLOCKS)

    def set_trainable_blocks(self, n):
        for p in self.parameters():
            p.requires_grad = False

        for block in list(self.model.encoder.layers)[-n:]:
            for p in block.parameters():
                p.requires_grad = True

        for p in self.model.encoder.ln.parameters():
            p.requires_grad = True

    def forward(self, x):
        z = self.model(x)
        if z.ndim != 2 or z.shape[-1] != 768:
            raise RuntimeError(
                f"Expected ViT-B/16 768-D features, got {tuple(z.shape)}"
            )
        return F.normalize(z.float(), dim=-1)

class Pipeline4LikePathologyHead(nn.Module):
    """Causal pathology path of Pipeline 4's disentangler, without spurious/fusion branches."""
    def __init__(self, in_dim=768, shared_dim=256, causal_dim=128):
        super().__init__()
        self.shared = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, shared_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.LayerNorm(shared_dim),
        )
        self.causal = nn.Sequential(
            nn.Linear(shared_dim, causal_dim),
            nn.GELU(),
            nn.LayerNorm(causal_dim),
        )
        self.path_head = nn.Linear(causal_dim, N_CLASSES)

    def forward(self, h):
        s = self.shared(h)
        zc = F.normalize(self.causal(s), dim=-1)
        return self.path_head(zc)

class Pipeline4LikeMILModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = CleanImageNetViTEncoder()
        self.head = Pipeline4LikePathologyHead()

    def patch_logits(self, flat_patches):
        return self.head(self.encoder(flat_patches))

    def forward_bag(self, bags):
        # bags: [B,K,3,224,224]
        B, K = bags.shape[:2]
        logits = self.patch_logits(
            bags.flatten(0, 1)
        ).view(B, K, N_CLASSES)

        image_logits = (
            torch.logsumexp(logits, dim=1)
            - math.log(K)
        )

        return image_logits, logits

def masked_bce_with_logits(logits, targets, mask):
    raw = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none",
    )
    return (
        (raw * mask).sum()
        / mask.sum().clamp_min(1.0)
    )

def pairwise_rank_loss(scores, targets, mask):
    losses = []

    for c in range(N_CLASSES):
        known = mask[:, c] > 0.5
        pos = scores[
            known & (targets[:, c] >= 0.5),
            c,
        ]
        neg = scores[
            known & (targets[:, c] < 0.5),
            c,
        ]

        if len(pos) == 0 or len(neg) == 0:
            continue

        diffs = (
            pos[:, None] - neg[None, :]
        ).reshape(-1)

        if diffs.numel() > MAX_RANK_PAIRS_PER_CLASS:
            # Deterministic prefix cap; batch ordering itself is seeded.
            diffs = diffs[:MAX_RANK_PAIRS_PER_CLASS]

        losses.append(
            F.softplus(-diffs).mean()
        )

    if not losses:
        return scores.sum() * 0.0

    return torch.stack(losses).mean()

def compute_multilabel_metrics(y_true, scores, known_mask):
    y_true = np.asarray(y_true, dtype=np.float64)
    scores = np.asarray(scores, dtype=np.float64)
    known_mask = np.asarray(known_mask, dtype=np.float64)

    if (
        y_true.shape != scores.shape
        or y_true.shape != known_mask.shape
        or y_true.ndim != 2
        or y_true.shape[1] != N_CLASSES
    ):
        raise ValueError(
            f"Shape mismatch: y={y_true.shape}, scores={scores.shape}, mask={known_mask.shape}"
        )

    if not np.all(np.isfinite(scores)):
        raise ValueError("Non-finite pathology scores detected")

    per_class = {}
    aucs, aps, f1s, bals = [], [], [], []

    probabilities = 1.0 / (
        1.0 + np.exp(-np.clip(scores, -40.0, 40.0))
    )

    for c, pathology in enumerate(TARGET_CLASSES):
        known = known_mask[:, c] > 0.5
        yy = y_true[known, c].astype(int)
        ss = scores[known, c]
        pp = probabilities[known, c]

        if len(yy) < 2 or len(np.unique(yy)) < 2:
            auc = ap = f1 = bal = np.nan
        else:
            auc = roc_auc_score(yy, ss)
            ap = average_precision_score(yy, ss)
            pred = (pp >= 0.5).astype(int)
            f1 = f1_score(yy, pred, zero_division=0)
            bal = balanced_accuracy_score(yy, pred)

        per_class[pathology] = {
            "AUROC": float(auc) if np.isfinite(auc) else np.nan,
            "AUPRC": float(ap) if np.isfinite(ap) else np.nan,
            "F1_at_0.5": float(f1) if np.isfinite(f1) else np.nan,
            "balanced_accuracy_at_0.5": float(bal) if np.isfinite(bal) else np.nan,
            "n_known": int(known.sum()),
            "n_positive": int(yy.sum()),
        }

        aucs.append(auc)
        aps.append(ap)
        f1s.append(f1)
        bals.append(bal)

    return {
        "macro_AUROC": float(np.nanmean(aucs)),
        "macro_AUPRC": float(np.nanmean(aps)),
        "macro_F1_at_0.5": float(np.nanmean(f1s)),
        "macro_balanced_accuracy_at_0.5": float(np.nanmean(bals)),
        "per_class": per_class,
    }

@torch.no_grad()
def evaluate_model(model, loader, report_weak_patch_metrics):
    model.eval()

    image_scores = []
    image_y = []
    image_m = []
    image_names = []
    patient_ids = []

    patch_scores = []
    patch_y = []
    patch_m = []
    patch_ranks = []

    for bags, y, m, names, patients in loader:
        bags = bags.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        m = m.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            enabled=AMP,
            dtype=torch.float16,
        ):
            img_logits, p_logits = model.forward_bag(bags)

        image_scores.append(img_logits.float().cpu())
        image_y.append(y.float().cpu())
        image_m.append(m.float().cpu())
        image_names.extend(list(map(str, names)))
        patient_ids.extend(list(map(str, patients)))

        if report_weak_patch_metrics:
            B, K, C = p_logits.shape
            patch_scores.append(
                p_logits.float().cpu().reshape(B * K, C)
            )
            patch_y.append(
                y.float().cpu()[:, None, :]
                .expand(B, K, C)
                .reshape(B * K, C)
            )
            patch_m.append(
                m.float().cpu()[:, None, :]
                .expand(B, K, C)
                .reshape(B * K, C)
            )
            patch_ranks.extend(
                list(range(K)) * B
            )

    S = torch.cat(image_scores).numpy()
    Y = torch.cat(image_y).numpy()
    M = torch.cat(image_m).numpy()

    image_metrics = compute_multilabel_metrics(
        Y, S, M
    )

    weak_patch_metrics = None
    rank_metrics = {}

    if report_weak_patch_metrics:
        PS = torch.cat(patch_scores).numpy()
        PY = torch.cat(patch_y).numpy()
        PM = torch.cat(patch_m).numpy()
        R = np.asarray(patch_ranks, dtype=int)

        weak_patch_metrics = compute_multilabel_metrics(
            PY, PS, PM
        )

        for rank in range(TOPK_PATCHES_PER_IMAGE):
            idx = R == rank
            rank_metrics[rank] = compute_multilabel_metrics(
                PY[idx], PS[idx], PM[idx]
            )

    return {
        "image_metrics": image_metrics,
        "weak_patch_metrics": weak_patch_metrics,
        "rank_metrics": rank_metrics,
        "image_names": image_names,
        "patient_ids": np.asarray(patient_ids, dtype=object),
        "y_true": Y,
        "known_mask": M,
        "scores": S,
    }

def make_model(seed):
    seed_everything(seed)
    model = Pipeline4LikeMILModel().to(DEVICE)

    trainable_encoder = sum(
        p.numel()
        for p in model.encoder.parameters()
        if p.requires_grad
    )
    trainable_head = sum(
        p.numel()
        for p in model.head.parameters()
        if p.requires_grad
    )

    print(
        f"Trainable encoder params={trainable_encoder:,} | "
        f"pathology-head params={trainable_head:,}"
    )

    if trainable_encoder <= 0 or trainable_head <= 0:
        raise RuntimeError("Expected trainable encoder tail and pathology head")

    return model

def train_one_seed(
    seed,
    train_dataset,
    val_dataset,
    report_weak_patch_metrics,
):
    model = make_model(seed)

    train_loader = make_loader(
        train_dataset,
        IMAGE_BATCH_SIZE,
        True,
        seed,
    )

    val_loader = make_loader(
        val_dataset,
        EVAL_IMAGE_BATCH_SIZE,
        False,
        seed + 10000,
    )

    encoder_params = [
        p for p in model.encoder.parameters()
        if p.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        [
            {
                "params": encoder_params,
                "lr": BACKBONE_LR,
            },
            {
                "params": list(model.head.parameters()),
                "lr": HEAD_LR,
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=AMP,
    )

    best_auc = -np.inf
    best_state = None
    history = []

    for epoch in range(1, SOURCE_EPOCHS + 1):
        model.train()
        running = 0.0
        n_batches = 0

        progress = tqdm(
            train_loader,
            desc=(
                f"{ARM_NAME} seed={seed} "
                f"epoch={epoch}/{SOURCE_EPOCHS}"
            ),
            leave=False,
        )

        for bags, y, m, _names, _patients in progress:
            bags = bags.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            m = m.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type="cuda",
                enabled=AMP,
                dtype=torch.float16,
            ):
                image_logits, _patch_logits = model.forward_bag(bags)

                bce = masked_bce_with_logits(
                    image_logits,
                    y,
                    m,
                )

                rank = pairwise_rank_loss(
                    image_logits,
                    y,
                    m,
                )

                loss = (
                    bce
                    + TARGET_AUC_RANK_WEIGHT * rank
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                [
                    p
                    for group in optimizer.param_groups
                    for p in group["params"]
                ],
                5.0,
            )

            scaler.step(optimizer)
            scaler.update()

            running += float(loss.detach().cpu())
            n_batches += 1

            progress.set_postfix(
                loss=f"{float(loss.detach().cpu()):.4f}"
            )

        val_result = evaluate_model(
            model,
            val_loader,
            report_weak_patch_metrics=False,
        )

        val_auc = val_result[
            "image_metrics"
        ]["macro_AUROC"]

        val_ap = val_result[
            "image_metrics"
        ]["macro_AUPRC"]

        row = {
            "arm": ARM_NAME,
            "seed": seed,
            "epoch": epoch,
            "train_loss": running / max(1, n_batches),
            "val_macro_AUROC": val_auc,
            "val_macro_AUPRC": val_ap,
        }

        history.append(row)
        print(row)

        if val_auc > best_auc:
            best_auc = float(val_auc)
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

    if best_state is None:
        raise RuntimeError("No best validation checkpoint was captured")

    model.load_state_dict(best_state, strict=True)

    pd.DataFrame(history).to_csv(
        OUT_DIR / f"{ARM_NAME}_seed{seed}_learning_curve.csv",
        index=False,
    )

    torch.save(
        {
            "model_state_dict": best_state,
            "arm": ARM_NAME,
            "seed": seed,
            "best_val_macro_AUROC": best_auc,
            "task_definition": TASK_DEFINITION,
            "auroc_definition": AUROC_DEFINITION,
            "settings": {
                "patch_count": TOPK_PATCHES_PER_IMAGE,
                "aggregation": IMAGE_AGGREGATION,
                "rank_weight": TARGET_AUC_RANK_WEIGHT,
                "backbone_lr": BACKBONE_LR,
                "head_lr": HEAD_LR,
                "weight_decay": WEIGHT_DECAY,
                "epochs": SOURCE_EPOCHS,
                "unfreeze_blocks": UNFREEZE_BLOCKS,
            },
        },
        OUT_DIR / f"{ARM_NAME}_seed{seed}_best.pt",
    )

    return model, best_auc

def prediction_dataframe(result):
    frame = pd.DataFrame({
        "image_name": list(map(str, result["image_names"])),
        "patient_id": list(map(str, result["patient_ids"])),
    })

    for c, pathology in enumerate(TARGET_CLASSES):
        frame[f"y_{pathology}"] = result["y_true"][:, c]
        frame[f"known_{pathology}"] = result["known_mask"][:, c]
        frame[f"score_{pathology}"] = result["scores"][:, c]
        frame[f"prob_{pathology}"] = (
            1.0
            / (
                1.0
                + np.exp(
                    -np.clip(
                        result["scores"][:, c],
                        -40.0,
                        40.0,
                    )
                )
            )
        )

    return frame


# -------------------------------------------------------------------------------------------------
# PREPARE ARM DATA
# -------------------------------------------------------------------------------------------------
if WHOLE_IMAGE_ARM:
    TRAIN_DATA = WholeImageBagDataset(train_df, train=True)
    VAL_DATA = WholeImageBagDataset(val_df, train=False)
    TEST_DATA = WholeImageBagDataset(test_df, train=False)
    REPORT_WEAK_PATCH_METRICS = False

else:
    print("\n" + "=" * 110)
    print("CHEXZERO PATCH MINING")
    print("=" * 110)

    chexzero, chexzero_preprocess = load_chexzero()

    POSITIVE_MATRIX = build_prompt_matrix(
        chexzero,
        ZS_POS,
    )
    NEGATIVE_MATRIX = build_prompt_matrix(
        chexzero,
        ZS_NEG,
    )

    TRAIN_PATCH_TABLE = extract_train_patches_for_arm(
        train_df,
        chexzero,
        chexzero_preprocess,
        POSITIVE_MATRIX,
        NEGATIVE_MATRIX,
    )

    VAL_PATCH_TABLE = build_common_image_only_eval_patches(
        val_df,
        "val",
        chexzero,
        chexzero_preprocess,
        POSITIVE_MATRIX,
        NEGATIVE_MATRIX,
    )

    TEST_PATCH_TABLE = build_common_image_only_eval_patches(
        test_df,
        "test",
        chexzero,
        chexzero_preprocess,
        POSITIVE_MATRIX,
        NEGATIVE_MATRIX,
    )

    validate_patch_table_geometry(
        TRAIN_PATCH_TABLE,
        f"{ARM_NAME}/train",
    )
    validate_patch_table_geometry(
        VAL_PATCH_TABLE,
        f"{ARM_NAME}/val",
    )
    validate_patch_table_geometry(
        TEST_PATCH_TABLE,
        f"{ARM_NAME}/test",
    )

    TRAIN_DATA = PatchBagDataset(
        TRAIN_PATCH_TABLE,
        train=True,
    )
    VAL_DATA = PatchBagDataset(
        VAL_PATCH_TABLE,
        train=False,
    )
    TEST_DATA = PatchBagDataset(
        TEST_PATCH_TABLE,
        train=False,
    )
    REPORT_WEAK_PATCH_METRICS = True

    del chexzero, POSITIVE_MATRIX, NEGATIVE_MATRIX
    gc.collect()
    torch.cuda.empty_cache()

if len(TRAIN_DATA) != len(train_df):
    raise RuntimeError(
        f"Training image count mismatch: dataset={len(TRAIN_DATA)} cohort={len(train_df)}"
    )
if len(VAL_DATA) != len(val_df):
    raise RuntimeError(
        f"Validation image count mismatch: dataset={len(VAL_DATA)} cohort={len(val_df)}"
    )
if len(TEST_DATA) != len(test_df):
    raise RuntimeError(
        f"Test image count mismatch: dataset={len(TEST_DATA)} cohort={len(test_df)}"
    )

print(
    f"Prepared {ARM_NAME}: "
    f"train_images={len(TRAIN_DATA):,}, "
    f"val_images={len(VAL_DATA):,}, "
    f"test_images={len(TEST_DATA):,}"
)

# -------------------------------------------------------------------------------------------------
# FINAL 5-SEED RUNS — same seeds in every notebook
# -------------------------------------------------------------------------------------------------
SEED_ROWS = []
PER_CLASS_ROWS = []
WEAK_PATCH_ROWS = []
PATCH_RANK_ROWS = []
PREDICTION_STORE = []

for seed in FINAL_SEEDS:
    marker_path = OUT_DIR / f"{ARM_NAME}_seed{seed}_complete.json"
    pred_path = OUT_DIR / f"{ARM_NAME}_seed{seed}_test_predictions.csv"

    if marker_path.exists() and pred_path.exists():
        saved = json.loads(
            marker_path.read_text(encoding="utf-8")
        )

        print(f"[resume] {ARM_NAME} seed={seed} already complete")

        SEED_ROWS.append(saved["seed_row"])
        PER_CLASS_ROWS.extend(saved["per_class_rows"])

        if saved.get("weak_patch_row") is not None:
            WEAK_PATCH_ROWS.append(saved["weak_patch_row"])

        PATCH_RANK_ROWS.extend(
            saved.get("patch_rank_rows", [])
        )

        PREDICTION_STORE.append(
            pd.read_csv(pred_path)
        )
        continue

    print("\n" + "=" * 110)
    print(f"FINAL TRAINING | {ARM_NAME} | seed={seed}")
    print("=" * 110)

    model, best_val_auc = train_one_seed(
        seed=seed,
        train_dataset=TRAIN_DATA,
        val_dataset=VAL_DATA,
        report_weak_patch_metrics=REPORT_WEAK_PATCH_METRICS,
    )

    test_loader = make_loader(
        TEST_DATA,
        EVAL_IMAGE_BATCH_SIZE,
        False,
        seed + 20000,
    )

    result = evaluate_model(
        model,
        test_loader,
        report_weak_patch_metrics=REPORT_WEAK_PATCH_METRICS,
    )

    image_metrics = result["image_metrics"]

    pred_df = prediction_dataframe(result)
    pred_df.to_csv(pred_path, index=False)
    PREDICTION_STORE.append(pred_df)

    seed_row = {
        "arm": ARM_NAME,
        "seed": seed,
        "best_val_macro_AUROC": best_val_auc,
        "test_image_macro_AUROC": image_metrics["macro_AUROC"],
        "test_image_macro_AUPRC": image_metrics["macro_AUPRC"],
        "test_image_macro_F1_at_0.5": image_metrics["macro_F1_at_0.5"],
        "test_image_macro_balanced_accuracy_at_0.5": (
            image_metrics["macro_balanced_accuracy_at_0.5"]
        ),
    }

    per_class_rows = []

    for pathology in TARGET_CLASSES:
        per_class_rows.append({
            "arm": ARM_NAME,
            "seed": seed,
            "pathology": pathology,
            **image_metrics["per_class"][pathology],
        })

    weak_patch_row = None
    patch_rank_rows = []

    if REPORT_WEAK_PATCH_METRICS:
        weak_patch_metrics = result["weak_patch_metrics"]

        weak_patch_row = {
            "arm": ARM_NAME,
            "seed": seed,
            "analysis_role": (
                "secondary weak image-label patch diagnostic; "
                "not localization accuracy"
            ),
            "macro_AUROC": weak_patch_metrics["macro_AUROC"],
            "macro_AUPRC": weak_patch_metrics["macro_AUPRC"],
        }

        for rank, metrics in result["rank_metrics"].items():
            patch_rank_rows.append({
                "arm": ARM_NAME,
                "seed": seed,
                "rank": int(rank),
                "analysis_role": (
                    "secondary weak image-label patch-rank diagnostic"
                ),
                "macro_AUROC": metrics["macro_AUROC"],
                "macro_AUPRC": metrics["macro_AUPRC"],
            })

    SEED_ROWS.append(seed_row)
    PER_CLASS_ROWS.extend(per_class_rows)

    if weak_patch_row is not None:
        WEAK_PATCH_ROWS.append(weak_patch_row)

    PATCH_RANK_ROWS.extend(patch_rank_rows)

    marker_path.write_text(
        json.dumps(
            {
                "seed_row": seed_row,
                "per_class_rows": per_class_rows,
                "weak_patch_row": weak_patch_row,
                "patch_rank_rows": patch_rank_rows,
            },
            indent=2,
            default=float,
        ),
        encoding="utf-8",
    )

    print(
        f"TEST {ARM_NAME} seed={seed}: "
        f"image macro AUROC={image_metrics['macro_AUROC']:.4f} | "
        f"image macro AUPRC={image_metrics['macro_AUPRC']:.4f}"
    )

    if REPORT_WEAK_PATCH_METRICS:
        print(
            f"WEAK PATCH {ARM_NAME} seed={seed}: "
            f"macro AUROC={result['weak_patch_metrics']['macro_AUROC']:.4f} | "
            f"macro AUPRC={result['weak_patch_metrics']['macro_AUPRC']:.4f}"
        )

    del model, test_loader
    gc.collect()
    torch.cuda.empty_cache()

SEED_METRICS = pd.DataFrame(SEED_ROWS)
PER_CLASS_METRICS = pd.DataFrame(PER_CLASS_ROWS)
WEAK_PATCH_METRICS = pd.DataFrame(WEAK_PATCH_ROWS)
PATCH_RANK_METRICS = pd.DataFrame(PATCH_RANK_ROWS)

SEED_METRICS.to_csv(
    OUT_DIR / "per_seed_image_multilabel_metrics.csv",
    index=False,
)
PER_CLASS_METRICS.to_csv(
    OUT_DIR / "per_seed_per_class_multilabel_metrics.csv",
    index=False,
)

if len(WEAK_PATCH_METRICS):
    WEAK_PATCH_METRICS.to_csv(
        OUT_DIR / "per_seed_weak_patch_metrics.csv",
        index=False,
    )

if len(PATCH_RANK_METRICS):
    PATCH_RANK_METRICS.to_csv(
        OUT_DIR / "per_seed_patch_rank_metrics.csv",
        index=False,
    )

SUMMARY = pd.DataFrame([{
    "arm": ARM_NAME,
    "n_seeds": len(SEED_METRICS),
    "image_macro_AUROC_mean": SEED_METRICS[
        "test_image_macro_AUROC"
    ].mean(),
    "image_macro_AUROC_sd": SEED_METRICS[
        "test_image_macro_AUROC"
    ].std(ddof=1),
    "image_macro_AUPRC_mean": SEED_METRICS[
        "test_image_macro_AUPRC"
    ].mean(),
    "image_macro_AUPRC_sd": SEED_METRICS[
        "test_image_macro_AUPRC"
    ].std(ddof=1),
    "image_macro_F1_at_0.5_mean": SEED_METRICS[
        "test_image_macro_F1_at_0.5"
    ].mean(),
    "image_macro_balanced_accuracy_at_0.5_mean": SEED_METRICS[
        "test_image_macro_balanced_accuracy_at_0.5"
    ].mean(),
}])

if len(WEAK_PATCH_METRICS):
    SUMMARY["weak_patch_macro_AUROC_mean"] = WEAK_PATCH_METRICS[
        "macro_AUROC"
    ].mean()
    SUMMARY["weak_patch_macro_AUROC_sd"] = WEAK_PATCH_METRICS[
        "macro_AUROC"
    ].std(ddof=1)
    SUMMARY["weak_patch_macro_AUPRC_mean"] = WEAK_PATCH_METRICS[
        "macro_AUPRC"
    ].mean()
    SUMMARY["weak_patch_macro_AUPRC_sd"] = WEAK_PATCH_METRICS[
        "macro_AUPRC"
    ].std(ddof=1)

SUMMARY.to_csv(
    OUT_DIR / "paper_table_summary.csv",
    index=False,
)

PER_CLASS_SUMMARY = (
    PER_CLASS_METRICS
    .groupby(["arm", "pathology"], as_index=False)
    .agg(
        AUROC_mean=("AUROC", "mean"),
        AUROC_sd=("AUROC", "std"),
        AUPRC_mean=("AUPRC", "mean"),
        AUPRC_sd=("AUPRC", "std"),
    )
)

PER_CLASS_SUMMARY.to_csv(
    OUT_DIR / "paper_table_per_class_mean_sd.csv",
    index=False,
)

# -------------------------------------------------------------------------------------------------
# CROSSED TRAINING-SEED x TEST-PATIENT BOOTSTRAP
# -------------------------------------------------------------------------------------------------
def aligned_prediction_arrays(prediction_dfs):
    base = (
        prediction_dfs[0]
        .sort_values("image_name")
        .reset_index(drop=True)
    )

    names = base["image_name"].astype(str).to_numpy()
    patients = base["patient_id"].astype(str).to_numpy()

    y_cols = [f"y_{p}" for p in TARGET_CLASSES]
    m_cols = [f"known_{p}" for p in TARGET_CLASSES]
    s_cols = [f"score_{p}" for p in TARGET_CLASSES]

    y_true = base[y_cols].to_numpy(float)
    known_mask = base[m_cols].to_numpy(float)

    score_stack = []

    for df in prediction_dfs:
        ordered = (
            df.set_index("image_name")
            .loc[names]
            .reset_index()
        )

        if not np.array_equal(
            ordered[y_cols].to_numpy(float),
            y_true,
        ):
            raise RuntimeError("Prediction label mismatch across seeds")

        if not np.array_equal(
            ordered[m_cols].to_numpy(float),
            known_mask,
        ):
            raise RuntimeError("Prediction mask mismatch across seeds")

        if not np.array_equal(
            ordered["patient_id"].astype(str).to_numpy(),
            patients,
        ):
            raise RuntimeError("Prediction patient mismatch across seeds")

        score_stack.append(
            ordered[s_cols].to_numpy(float)
        )

    return (
        y_true,
        known_mask,
        patients,
        np.stack(score_stack, axis=0),
    )

def valid_patient_bootstrap_indices(
    y_true,
    known_mask,
    patient_ids,
    rng,
    max_attempts=2000,
):
    unique_patients = np.unique(patient_ids)

    for _ in range(max_attempts):
        sampled_patients = rng.choice(
            unique_patients,
            size=len(unique_patients),
            replace=True,
        )

        idx = np.concatenate([
            np.where(patient_ids == p)[0]
            for p in sampled_patients
        ])

        valid = True

        for c in range(N_CLASSES):
            known = known_mask[idx, c] > 0.5
            values = y_true[idx, c][known]

            if known.sum() < 2 or len(np.unique(values)) < 2:
                valid = False
                break

        if valid:
            return idx.astype(np.int64)

    raise RuntimeError(
        "Could not draw a patient bootstrap replicate with binary support for all classes"
    )

def crossed_seed_patient_bootstrap(
    prediction_dfs,
    reps,
    seed,
):
    y_true, known_mask, patient_ids, scores = (
        aligned_prediction_arrays(prediction_dfs)
    )

    rng = np.random.RandomState(seed)
    n_seeds = scores.shape[0]

    auc_values = []
    ap_values = []

    for _ in range(reps):
        sampled_seed_positions = rng.choice(
            np.arange(n_seeds),
            size=n_seeds,
            replace=True,
        )

        patient_idx = valid_patient_bootstrap_indices(
            y_true,
            known_mask,
            patient_ids,
            rng,
        )

        seed_auc = []
        seed_ap = []

        for seed_pos in sampled_seed_positions:
            metrics = compute_multilabel_metrics(
                y_true[patient_idx],
                scores[seed_pos, patient_idx],
                known_mask[patient_idx],
            )

            seed_auc.append(metrics["macro_AUROC"])
            seed_ap.append(metrics["macro_AUPRC"])

        auc_values.append(np.nanmean(seed_auc))
        ap_values.append(np.nanmean(seed_ap))

    auc_values = np.asarray(auc_values, dtype=float)
    ap_values = np.asarray(ap_values, dtype=float)

    return {
        "arm": ARM_NAME,
        "bootstrap_reps": int(reps),
        "macro_AUROC_bootstrap_mean": float(np.nanmean(auc_values)),
        "AUROC_CI_low": float(np.nanpercentile(auc_values, 2.5)),
        "AUROC_CI_high": float(np.nanpercentile(auc_values, 97.5)),
        "macro_AUPRC_bootstrap_mean": float(np.nanmean(ap_values)),
        "AUPRC_CI_low": float(np.nanpercentile(ap_values, 2.5)),
        "AUPRC_CI_high": float(np.nanpercentile(ap_values, 97.5)),
    }

BOOTSTRAP_TABLE = pd.DataFrame([
    crossed_seed_patient_bootstrap(
        PREDICTION_STORE,
        BOOTSTRAP_REPS,
        BOOTSTRAP_SEED,
    )
])

BOOTSTRAP_TABLE.to_csv(
    OUT_DIR / "paper_table_crossed_seed_patient_bootstrap_95CI.csv",
    index=False,
)

# -------------------------------------------------------------------------------------------------
# PROVENANCE + CLAIM BOUNDARY
# -------------------------------------------------------------------------------------------------
TOTAL_WALL_CLOCK_SECONDS = float(
    time.perf_counter() - RUN_START_TIME
)

PROVENANCE = {
    "arm": ARM_NAME,
    "arm_config": ARM_CONFIG,
    "schema": EXPERIMENT_SCHEMA,
    "task_definition": TASK_DEFINITION,
    "auroc_definition": AUROC_DEFINITION,
    "patch_metric_scope": PATCH_METRIC_SCOPE,
    "dataset_evaluated": "NIH",
    "multi_positive_retained": True,
    "unknown_labels_masked": True,
    "classifier_output": "five independent pathology logits",
    "classifier_probability": "independent sigmoid per pathology",
    "loss": "masked BCEWithLogitsLoss + 0.40 pairwise ranking loss",
    "patch_count": TOPK_PATCHES_PER_IMAGE,
    "image_aggregation": IMAGE_AGGREGATION,
    "representation_backbone": "torchvision ViT-B/16 IMAGENET1K_V1",
    "trainable_backbone_tail_blocks": UNFREEZE_BLOCKS,
    "backbone_lr": BACKBONE_LR,
    "head_lr": HEAD_LR,
    "weight_decay": WEIGHT_DECAY,
    "epochs": SOURCE_EPOCHS,
    "patch_budget_batch": BATCH_PATCH_BUDGET,
    "image_batch_size": IMAGE_BATCH_SIZE,
    "validation_test_reports_used": False,
    "validation_test_labels_used_for_patch_selection": False,
    "chexzero_role": "patch selection only" if NEEDS_CHEXZERO else "not used",
    "full_pipeline4_fusion_present": False,
    "quantum_component_present": False,
    "cohort_hash": COHORT_HASH,
    "extraction_hash": EXTRACTION_HASH if not WHOLE_IMAGE_ARM else "not_applicable",
    "final_seeds": FINAL_SEEDS,
    "bootstrap_reps": BOOTSTRAP_REPS,
    "chexzero_checkpoint_sha256": CHEXZERO_SHA256,
    "clip_module_sha256": CLIP_MODULE_SHA256,
    "total_wall_clock_seconds": TOTAL_WALL_CLOCK_SECONDS,
}

(OUT_DIR / "provenance.json").write_text(
    json.dumps(PROVENANCE, indent=2, default=str),
    encoding="utf-8",
)

(OUT_DIR / "CLAIM_BOUNDARY.txt").write_text(
    "PRIMARY: NIH image-level five-pathology multilabel AUROC/AUPRC using independent pathology scores.\n"
    "SECONDARY: patch/rank AUROC/AUPRC uses inherited image labels and is NOT lesion-localization accuracy.\n"
    "PIPELINE-4 MATCH: label formulation, binary AUROC semantics, 6-patch MIL aggregation, "
    "masked BCE + rank loss, ImageNet ViT-B/16 last-2-block adaptation, LR/WD/epochs.\n"
    "NOT COPIED: CheXpert target adaptation, prototypes, few-shot support, fusion, or quantum branches.\n"
    "Therefore do NOT numerically equate these NIH AUROCs with Pipeline 4 CheXpert fusion AUROCs.\n",
    encoding="utf-8",
)

print("\n" + "=" * 110)
print(f"FINAL PAPER SUMMARY — {ARM_NAME}")
print("=" * 110)
print(SUMMARY.to_string(index=False))

print("\nPer-class image-level AUROC/AUPRC:")
print(PER_CLASS_SUMMARY.to_string(index=False))

print("\nCrossed seed x patient 95% CI:")
print(BOOTSTRAP_TABLE.to_string(index=False))

if len(WEAK_PATCH_METRICS):
    print("\nWeak patch metrics (NOT localization accuracy):")
    print(WEAK_PATCH_METRICS.to_string(index=False))

print("\nInterpretation:")
print(PATCH_METRIC_SCOPE)

print(
    f"\nTotal wall-clock time: "
    f"{TOTAL_WALL_CLOCK_SECONDS / 3600.0:.2f} hours"
)
print("Saved experiment directory:", OUT_DIR)
